<div style="float: right; font-size: 15pt;">COM-202, <a href="https://www.epfl.ch/">EPFL</a></div>
<div style="clear: both;"></div>
<div style="float: right; font-size: 10pt;"><a href="https://people.epfl.ch/paolo.prandoni">Paolo Prandoni</a></div>

<h1 style="font-size: 60pt; color: #B51F1F;">One Bit at a Time</h1>
<h2  style="margin-top: -20px; color: #B51F1F; text-transform: none; ">the amazing audio tricks of 1980s videogames</h2>

![pacman](img/pacman.gif)

In [ ]:
import numpy as np
import scipy.signal as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as tck
import ipywidgets as wg
from fractions import Fraction

from scipy.io import wavfile
from IPython.display import Audio, YouTubeVideo, clear_output

In [ ]:
plt.rcParams["figure.figsize"] = (14,4)

In [ ]:
SF = 48000

def play(s, sf=SF, auto=False):
    return display(Audio(s, rate=sf, autoplay=auto))

def multiplay(files, labels=None, sf=SF):
    outs = [ wg.Output() for f in files ]
    for n in range(len(files)):
        with outs[n]:
            if labels is not None:
                print(labels[n])
            display(Audio(files[n], rate=sf))
    display(wg.HBox(outs))

# 1-bit audio

<img src="img/dsd.png" style="float: right; width: 150px; margin: 00px 30px;"/>

the term "1-bit audio" means different things to different people:

 * audio engineers will immediately think of sigma-delta converters, the DSD standard, or SuperAudio CDs
 * but a few people will think of the *chiptune* scene, retro-computing, and some avant-garde electronic music

both are correct and, interestingly, closely related!

# outline of this lecture

 * what is 1-bit music?
 * a short history of videogame audio
 * making sounds in early home computers
 * binary waveforms, polyphony, and PDM
 * the road to sigma-delta quantization

# some definitions

## chiptune

> chiptune (aka *8-bit music*) is an electronic music genre created using the intrinsically limited sound capabilities of the vintage audio chips used in early video game consoles, home computers, and arcade machines

## 1-bit music

> a niche subgenre of chiptune where the sound capabilities are even more limited and can only produce two-level (binary) waveforms

# 1-bit music todays

<img src="img/one_bit_symphony.jpg" style="float: right; height: 300px; margin: 20px;"/>
<img src="img/1bitmusic.jpg" style="float: left; height: 300px; margin: 20px;"/>

In [ ]:
multiplay(["snd/Perich_mov2.wav", "snd/Perich_mov3.wav", ])

# audio in early video games

 * the very first video games had no sound at all
 * 1972, **Pong** -  simple beeps when the ball bounces
 * 1978, **Space Invaders** - continous in-game sound
 * 1980,  **PacMan** - the sound effects composer (Toshio Kai) is credited

in all cases (arcade, consoles, home computers) sound was generated by dedicated audio chips

<img src="img/pong.jpg" style="float: left; height: 100px; margin: 55px 50px 0px;"/>
<img src="img/spaceinvaders.png" style="float: left; height: 210px; margin: 20px 50px 0px;"/>
<img src="img/pacman_screen.jpg" style="float: left; height: 250px; margin: 0px 0px 0px 50px;"/>

# chiptune 

 * audio chips contained two or more programmable oscillators
 * each chip was an idiosyncratic little musical instruments (famously, well-known design bugs had to be left as-is in subsequent releases so as not to change the special "tone" of the chip)
 * in the hands of creative composers, these chips could produce pretty amazing soundtracks
 * the chiptune scene is still very active today

In [ ]:
YouTubeVideo(id='kJgdw1Lhzg4', width=400) 

# the home computing era

<img src="img/homepc.jpg" style="float: right; height: 400px; margin: -10px 0 0 20px;"/>

the early home computers were advertised as "educational" tools but in the end people bought them primarily to play videogames

 * 1977: Apple I
 * 1982: ZX Spectrum, Commodore64
 * 1984: Macintosh
 * 1987: Amiga500

home computers also used dedicated audio chips - with one notable exception that we will look at in detail

# the transition to digital audio

<img src="img/soundblaster.png" style="float: right; height: 200px; margin: 0px;"/>

dedicated audio chips were progressively replaced by the *soundcard*, a general-purpose D/A converter that could play arbitrary digital audio files stored in memory

 * compact disks entered the market in the early 1980s
 * 1988: first Soundblaster card (ISA, **8 bits**, 22.1 kHz)
 * 1992: CD-quality soundcards (ISA, **16 bits**, 44.1 kHz)
 * 1998: fast **PCI** soundcards finally rendered audio chips obsolete

# the notable exception: Sinclair's ZX Spectrum

 * released in 1983, it featured an 8-bit CPU (a Zilog Z80 at 3.5 MHz) and up to 48 KBytes of RAM
 * it did not have a sound chip, but the CPU could drive the small onboard speaker directly using a single data line (on or off)
 * technically, this was equivalent to a D/A converter... but the resolution was just one bit per sample!

![spectrum](img/spectrum.png)

# digital audio in one slide

<img src="img/dac_example.png" style="float: right; height: 300px; margin: -50px 0 0 20px;"/>

 * a digital signal is a sequence of samples encoded over $R$ bits each
 * to drive a loudspeaker we need an analog voltage signal
 * to transform the sequence of digital samples into an analog signal we use a D/A converter: 
     * the number of samples per second "consumed" by the D/A is called the interpolation rate $F_s$
     * the maximum analog frequency that a D/A can produce is $F_s/2~\text{Hz}$
     * the dynamic range of the analog signal depends on the bitrate $R$:
       $$
           \text{DR} = 20 \log_{10}\left(\frac{x_{\max}}{x_{\min}}\right) = 20 \log_{10}\left(\frac{2^R-1}{1}\right) \approx 6R~\text{dB}
       $$
 * a D/A circuit needs about $2R$ precision resistors so its manufacturing cost grows with the bitrate

<img src="img/dac3bit.jpg" style="float: left; height: 250px; margin: 50px 0 0 120px;"/>

# pitch, timbre, loudness

if an audio signal contains a repeating pattern, we perceive a musical *pitch*

 * pitch is determined by how many times per second the pattern repeats
 * the shape of the repeating pattern does not affect the pitch
 * the shape plays a role in other sound qualities:
   * _timbre_: a smooth shape will sound full and "round", while a jagged pattern will sound thin and "reedy"
   * _loudness_: the shape's RMS influences in complicated ways the perceived loudness
    $$
        x_{\text{RMS}} = \sqrt{\frac{1}{P-1}\sum_{k=0}^{P-1} |x[n]|^2}
    $$


In [ ]:
 def pitch_demo(sf=SF):    
    f = wg.IntSlider(value=330, min=20, max=1500, step=10, description='freq (Hz)')
    d = wg.widgets.Dropdown(options=['sine', 'sawtooth', 'square', 'random'], value='sine', description='pattern')
    b = wg.Checkbox(description='redo')
    player, text = wg.Output(), wg.Output() 
    with text:
        print('\t\t\t')
    
    def render(f, shape, redo, t=0.1):
        P = int(sf / f + 0.5)
        N = int(t * sf / P) * P
        if shape == 'sine':
            p = np.sin(2 * np.pi / P * np.arange(P))
        elif shape == 'sawtooth':
            p = np.linspace(-1, 1, P)
        elif shape == 'square':
            p = np.r_[np.ones(P // 2), -np.ones(P - P // 2)]
        elif shape == 'random':
            p = 2 * np.random.rand(P) - 1
        plt.subplot(2,3,2)
        plt.stem(p)
        plt.xticks(np.arange(0, P, max(1, P // 10)))
        plt.yticks([-1, 1])
        plt.xlabel(f'power: {np.mean(p ** 2):.2f}')
        plt.subplot(2,1,2)
        plt.plot(np.linspace(0, t, N), np.resize(p, N))
        plt.xlabel('seconds')
        plt.yticks([0, 1])
        plt.tight_layout()
        with player:
            clear_output()
            display(Audio(np.resize(p, sf), rate=sf, autoplay=False))
            
    ge = wg.interactive_output(render, {'f': f, 'shape': d, 'redo': b})
    display(wg.VBox([ ge, wg.HBox([ f, d, b, text, player] ) ]))

In [ ]:
pitch_demo(sf=8000)

# binary waveforms

 * the simplest digital audio waveform is a sequence of samples that have only two possible values (binary signals)
 * key feature of binary signals: they do not require a D/A and can drive a loudspeaker directly

<img src="img/spectrum.png" style="height: 200px; margin: 10px 150px 10px  0;"/>


 * the Spectrum audio interface was essentially a D/A converter with:
    * very poor DR ($R=1$, i.e. $\text{DR} = 0~\text{dB}$)
    * but very high interpolation rate ($F_s \approx 200~\text{kHz}$)



In [ ]:
def square_wave(freq, duration, duty=50, sf=SF):
    period, half = int(sf / freq + 0.5), max(1, int(duty * sf / freq / 100))
    return np.resize(np.r_[ np.ones(half), np.zeros(period - half) ], int(duration * sf))

In [ ]:
def note_to_freq(note):
    try:
        f = float(note)
        return f
    except ValueError:
        # note name and octave to frequency
        C4 = 261.63
        SEMITONES = {'C': 0, 'C#': 1, 'Db': 1, 'D': 2, 'D#': 3, 'Eb': 3, 'E': 4, 'F': 5, 'F#': 6, 'Gb': 6, 
                     'G': 7, 'G#': 8, 'Ab': 8, 'A': 9, 'A#': 10, 'Bb': 10, 'B': 11}  
        try:
            s = SEMITONES[note[:-1]]
            octave = int(note[-1]) - 4        
            return C4 * (2 ** octave) * (2.0 ** (s / 12.0))
        except KeyError:
            return 0

# the BEEP command

 * the ZX Spectrum could generate **pitched binary square waves** via the BEEP command:
   * eg. `BEEP(0.5, 2)` would play for half a second a note pitched two semitones above middle C (i.e. D4) 
 * since the binary samples were generated and sent to the speaker directly by the CPU, the Spectrum would remain completely "frozen" while beeping
 * still, one could produce recognizable melodies:

In [ ]:
def BEEP(score, timing=1000, duty_cycle=50, sf=SF):
    s = []
    p = 0
    for (pitch, duration) in score:
        f, t = note_to_freq(pitch), timing * duration / 1000
        if f > 0:
            n = np.arange(0, int(duration*sf * timing/ 1000)) + p
            p = n[-1]
            s = np.r_[ s, square_wave(f, t, duty_cycle, sf) ]  #np.cos(2 * np.pi * f / sf * n) ] #
        else:
            s = np.r_[ s, np.zeros(int(t * sf)) ]
    return s

In [ ]:
pacman_melody = (
    ('B4', 2), ('B5', 2), ('F#5', 2), ('D#5', 2), ('B5', 1), ('F#5', 3), ('D#5', 4), 
    ('C5', 2), ('C6', 2), ('G5', 2),  ('E5', 2),  ('C6', 1), ('G5', 3),  ('E5', 4),
    ('B4', 2), ('B5', 2), ('F#5', 2), ('D#5', 2), ('B5', 1), ('F#5', 3), ('D#5', 4), 
    ('D#5', 1), ('E5', 1), ('F5', 1.9), (' ', 0.1), ('F5', 1), ('F#5', 1), ('G5', 1.9), (' ', 0.1), 
    ('G5', 1), ('G#5', 1), ('A5', 1.9), (' ', 0.1), ('B5', 4) )

play(BEEP(pacman_melody, timing=65))

![pacman](img/pacman.gif)

# binary square waves

 * pitch is determined by the length of the period $P$ (in samples): $f = F_s/P$
 * in a balanced wave, half the period is one and the other half zero
 * we can change the timbre a bit by modifying the _duty cycle_ $d = D/P$:
   * $D$ is the number of samples equal to one in a period
   * for a balanced wave, $d = 50\%$

In [ ]:
 def square_wave_demo(sf=SF):    
    f = wg.IntSlider(value=440, min=20, max=1500, step=10, description='freq (Hz)')
    d = wg.IntSlider(value=50, min=0, max=100, step=1, description='duty cyle (%)')
    player, text = wg.Output(), wg.Output() 
    
    def render(f, d, t=0.1):
        p = square_wave(f, 1/f, d, sf)
        P = len(p)
        N = int(t * sf / P) * P
        plt.subplot(2,3,2)
        plt.stem(p)
        plt.xticks(np.arange(0, P, max(1, P // 10)))
        plt.yticks([0, 1])
        plt.xlabel('samples')
        plt.subplot(2,1,2)
        plt.plot(np.linspace(0, t, N), np.resize(p, N))
        plt.xlabel('seconds')
        plt.yticks([0, 1])
        plt.tight_layout()
        with player:
            clear_output()
            display(Audio(square_wave(f, 1, d, sf), rate=sf, autoplay=False))
        with text:
            clear_output()
            print(f'\t period: {P} samples; actual frequency:{sf/P:.1f} Hz (F_s={sf} Hz)')
            
    ge = wg.interactive_output(render, {'f': f, 'd': d})
    display(wg.VBox([ ge, wg.HBox([ wg.VBox([ wg.HBox([ f, d ]) , text ]), player] ) ]))

In [ ]:
square_wave_demo(sf=8000)

# beyond simple BEEPs

not a lot that we can do with square waves but:
 * by changing the frequency over time we obtain a _chirp_
 * typically the frequency follows an exponential sweep: $f_{n+1} = \alpha f_n$

In [ ]:
def square_chirp(f_start=300, f_stop=1500, sweep_time=.1, duration=0, sf=SF):
    # start frequency, stop frequency, sweep time, total time
    L, p1, p2 = int(sf * sweep_time), int(sf / f_start), int(sf / f_stop)
    a = 1 - (p1 - p2) / L
    N = int(np.log(p2 / p1) / np.log(a) + 0.5)
    s = []
    for n in range(N):
        p = int(p1)
        s = np.r_[ s, np.r_[ np.ones(p // 2), np.zeros(p - p // 2) ] ]
        p1 = p1 * a
    M = len(s) if duration == 0 else int(duration * sf)
    return np.resize(s, M)

In [ ]:
plt.subplot(211); plt.plot(square_chirp(10, 220, 1, 0, 1000));

In [ ]:
multiplay([square_chirp(f_start=300, f_stop=1000, sweep_time=.8, duration=3), square_chirp(duration=1)], ['ALERT! ALERT!', 'FIRE THE LASER BEAMS!'])

# but what about polyphony?

 * dedicated audio chips had in at least two or three independent oscillators
 * most video game signature tunes were in fact polyphonic
 * in theory we can add binary waveforms together and obtain a polyphonic signal:

In [ ]:
pacman_bass = (
    ('B2', 6), ('B3', 2), ('B2', 6), ('B3', 2), ('C3', 6), ('C4', 2), ('C3', 6), ('C4', 2), 
    ('B2', 6), ('B3', 2), ('B2', 6), ('B3', 2), ('F#3', 4), ('G#3', 4), ('A#3', 4), ('B3', 4) )

pac_t = BEEP(pacman_melody, timing=65)
pac_b = BEEP(pacman_bass, timing=65)
pacman = (pac_t + pac_b) / 2
play(pacman)

![pacman](img/pacman.gif)

# the problem with binary polyphony

the sum of binary sgnals is no longer a binary signal!

 * the sum of $M$ binary signals spans $M+1$ distinct levels
 * if we mix $M$ binary signals via a normalized sum, the output will have $M-1$ intermediate levels between zero and one:
   \begin{align*}
       v_k[n] &\in \{0, 1\}, \quad k = 0, 1, \ldots, M-1 \\ \\
       \frac{1}{M}\sum_{k=0}^{M-1}v_k[n] &\in \left\{0, \frac{1}{M}, \frac{2}{M}, \ldots, \frac{M-1}{M}, 1 \right\}
   \end{align*}

In [ ]:
def show_voices(treble, bass, chunk, onebit=None):
    plt.subplot(411)
    plt.plot(treble[chunk], 'C2', label='treble')
    plt.xticks([])
    plt.yticks([0, 1])
    plt.ylim([-.5, 1.5])
    plt.legend()
    plt.subplot(412)
    plt.plot(bass[chunk], 'C7', label='bass')
    plt.xticks([])
    plt.yticks([0, 1])
    plt.ylim([-.5, 1.5])  
    plt.legend()
    plt.subplot(212)
    plt.plot(((treble + bass) / 2)[chunk], 'C11', lw=3, alpha=0.7, label='treble + bass')
    if onebit is not None:
        plt.plot(onebit[chunk], 'C0', lw=.5, alpha=.7, label='1-bit (offset by 0.5)')
    plt.yticks([0, 0.5, 1])
    plt.ylim([-.2, 1.2])   
    plt.legend()
    plt.tight_layout()

In [ ]:
pac_slice = slice(5800, 6850)
show_voices(pac_t, pac_b, pac_slice)

# the "game changer": Manic Miner

 * it would seem impossible to play polyphonic sounds with an on-off buzzer
 * but in 1984 this game came out

In [ ]:
YouTubeVideo(id='Us89iCZJuCQ', width=854, height=480, start=6) 

# followed by Vectron

 * a year later, this release featured concurrent melodies, percussive sounds and dynamics!


In [ ]:
YouTubeVideo(id='FySV12aWGkg', width=854, height=480, start=6) 

# how did they do it?

at the time, to me, these games seemed to have achieved the impossible: they could produce binary waveforms that, using only the onboard resources, i.e. the CPU and the cheap little speaker,

 * sounded polyphonic (up to five voices in Vectron)
 * had a bit of dynamic range (varying volume levels)


# reproducing Manic Miner in 2026
<img src="img/asm.png" style="float: right; width: 400px; margin: -30px 30px 30px;"/>

 * as I was preparing this lecture, I found a website with the complete Manic Miner disassembly
 * as a kid I used to be pretty good with Z80 machine language, so I translated the subroutine into Python

<br>

 * at this point it's probably too late to write to the author, but I did find a pretty ugly bug in the code! (more about this later)

let's start with the replica:

In [ ]:
# (duration, period_1, period_2)
manic_miner_intro = [
    (80,128,130), (80,102,103), (80,86,87), (50,86,87), (50,171,203), (50,43,51), (50,43,51), (50,171,203), (50,51,64), (50,51,64), (50,171,203), (50,128,129), 
    (50,128,129), (50,102,103), (50,86,87), (50,96,86), (50,171,192), (50,43,48), (50,43,48), (50,171,192), (50,48,68), (50,48,68), (50,171,192), (50,136,137), 
    (50,136,137), (50,114,115), (50,76,77), (50,76,77), (50,171,192), (50,38,48), (50,38,48), (50,171,192), (50,48,68), (50,48,68), (50,171,192), (50,136,137), 
    (50,136,137), (50,114,115), (50,76,77), (50,76,77), (50,171,203), (50,38,51), (50,38,51), (50,171,203), (50,51,64), (50,51,64), (50,171,203), (50,128,129), 
    (50,128,129), (50,102,103), (50,86,87), (50,64,65), (50,128,171), (50,32,43), (50,32,43), (50,128,171), (50,43,51), (50,43,51), (50,128,171), (50,128,129), 
    (50,128,129), (50,102,103), (50,86,87), (50,64,65), (50,128,152), (50,32,38), (50,32,38), (50,128,152), (50,38,48), (50,38,48), (50,256,256), (50,114,115), 
    (50,114,115), (50,96,97), (50,76,77), (50,76,153), (50,76,77), (50,76,77), (50,76,153), (50,91,92), (50,86,87), (50,51,205), (50,51,52), (50,51,52), 
    (50,51,205), (50,64,65), (50,102,103), (100,102,103), (50,114,115), (100,76,77), (50,86,87), (50,128,203), (25,128,256), (25,128,129), (50,128,203) ]

def play_mm_intro(score=manic_miner_intro, duty_cycle=50, sf=64000):
    rate = 64000 # playback at 64 kHz is closest in pitch to the original
    timing = 260 * sf / rate
    s, stop = np.zeros(sum(int(timing * item[0]) for item in score)), 0
    for item in score:
        start, stop = stop, stop + int(item[0] * timing)
        for period in item[1:]:
            duty = max(int(period * duty_cycle / 100), 1)
            for k in range(start, stop, period):
                s[k:k+duty] = 1
    return s, sf

In [ ]:
mm_original, mm_sf = play_mm_intro()
play(mm_original, mm_sf)

# how to achieve polyphony with binary signals

 * first we will explore a simple and intuitive way to mix binary waveforms, which however doesn't sound very good 
 * then we will look at **pulse density modulation (PDM)**, a much more powerful technique that can be used if the rate $F_s$ is much higher than the 20 kHz (the upper limit of the human hearing range):
   * PDM can be used on the ZX Spectrum, where $F_s \approx 200~\text{kHz}$
  
<img src="img/ATtiny85.jpg" style="float: right; width: 100px; margin: 10px 30px 10px 30px;"/>

   * today's 1-bit music players feature cheap microcontrollers such as the ATtiny85 (8 kB RAM, $F_s$ up to $250~\text{kHz}$)

most importantly, PDM is the key ingredient of **sigma-delta** encoding, the most common A/D and D/A technology  in use today 

# first idea: 1-bit polyphony via short duty cycles

 * the perceived pitch of a square wave does not depend on its duty cycle
 * the smallest duty cyle is one sample per period: in this case the waveform is called a _pulse train_
 * if we sum pulse trains together, we only have a problem if two pulses collide
 * this is probably a rare event and so, when pulses collide, we just round the result to $\{0, 1\}$:
   $$
       x[n] = \min\left\{\sum_{k=0}^{M-1}v_k[n], 1\right\}
   $$

In [ ]:
def pulse_train_pacman(duty_cycle=0, chunk=pac_slice):
    treble = BEEP(pacman_melody, timing=65, duty_cycle=duty_cycle)
    bass = BEEP(pacman_bass, timing=65, duty_cycle=duty_cycle) 

    show_voices(treble, bass, chunk)
    plt.show()
    
    mix = treble + bass
    overflow = mix > 1
    clipped_mix = np.copy(mix)
    clipped_mix[overflow] = 1 
    
    print(f'{int(100 * sum(overflow == True) / len(mix))}% collisions ({sum(overflow == True)} out of {len(mix)} samples) \n')
    multiplay([mix, clipped_mix], ['sum of voices', 'clipped sum'])

## let's try with PacMan

In [ ]:
pulse_train_pacman()

## but this doesn't sound very good

 * the problem with pulse trains is that they sound very "thin" (can't hear the bass)
 * this is because perceived loudness decreases with duty cycle
 * however, if we try to increase the duty cycle, the sound quality degrades fast

In [ ]:
pulse_train_pacman(30)

# here's a crazy idea:  polyphony via multiplexing

 * instead of averaging the inputs, create a signal by multiplexing them:
   $$
       x[n] = v_{(n \text{ mod } M)}[n] 
         = \begin{cases} v_0[n] & \text{if $n = 0, M, 2M, 3M, \ldots$} \\
         v_1[n] & \text{if $n = 1, M+1, 2M+1, 3M+1, \ldots$} \\
         \ldots \\
         v_{M-1}[n] & \text{if $n = M-1, 2M-1, 3M-1, \ldots$}
       \end{cases}
   $$
 * note that we are keeping just one sample out of $M$ in every input sequence!

here is an illustration of interleaving using two standard (non-binary) signals; the "shape" of the original signals is still visible in the multiplexed sequence

In [ ]:
def multiplexing_example(N = 20, binary=False):
    n = np.arange(2 * N + 1)
    if binary:
        yticks, ylim = [0, 1], [-0.2, 1.2]
        x1 = np.resize(np.r_[np.ones(3), np.zeros(4)], 2*N+1)
        x2 = np.resize(np.r_[np.ones(2), np.zeros(2)], 2*N+1)
    else:
        yticks, ylim = [-1, 0, 1], [-1.2, 1.2]
        x1 = (n - N) / N
        x2 = np.cos(2 * np.pi * n / N)
    plt.subplot(221)
    plt.stem(x1, 'C1')
    plt.yticks(yticks)
    plt.ylim(ylim)
    plt.subplot(223)
    plt.stem(x2, 'C2')
    plt.yticks(yticks)
    plt.ylim(ylim)
    plt.subplot(122)
    plt.stem(n[::2], x1[::2], 'C1')
    plt.stem(n[1::2], x2[1::2], 'C2')
    plt.yticks(yticks)
    plt.ylim(ylim)

In [ ]:
multiplexing_example()

on the other hand, multiplexing binary waveforms completely obfuscates the original pattern... will it work?

In [ ]:
multiplexing_example(binary=True)

the code for a one-bit multiplexing synthesizer is quite simple:

In [ ]:
def onebit_synth(score, timing=1000, duty_cycle=50, sf=SF):
    # multi-voice 1-bit synthesizer using voice interleaving
    num_voices, timing = len(score), timing * sf / 1000
    samples = np.max([ sum(int(timing * duration) for (_, duration) in voice) for voice in score ])
    s = np.zeros(samples)
    for voice_ix, voice in enumerate(score):
        ix = 0
        for (note, duration) in voice:    
            f = note_to_freq(note)
            N = int(timing * duration)
            if f > 0:                
                period = int((sf / f) + 0.5)
                duty = max(int(period * duty_cycle / 100), 1)
                for n in range(ix, ix + N):
                    if (n % num_voices) == voice_ix and ((n-ix) % period) < duty:
                        s[n] = 1
            ix += N
    return s

and we can use it to generate a "native" one-bit version of the polyphonic PacMan theme 

In [ ]:
pacman_onebit = onebit_synth([pacman_melody, pacman_bass], timing=65)
multiplay([pacman_onebit, pacman], ['1-bit multiplexed', 'original (3 levels)'])

# how does it work?

 * we are mixing two discrete-time binary signals
 * when the two inputs have the same value, both the additive mix and the multiplexed mix are equal to the signals' common value 
 * when the two inputs are different:
     * in the additive mix, the output will be the intermediate value $1/2$ (yielding a non-binary output)
     * in the multiplexed mix, the output will alternate between zero and one with every sample ($F_s$ times per second)

In [ ]:
show_voices(pac_t, pac_b, pac_slice, pacman_onebit)

In [ ]:
def show_pdm(multilevel, onebit, chunk, lowpass=None, delay=4):
    plt.subplot(211)
    plt.plot(multilevel[chunk], 'C11', lw=4, alpha=0.6, label='multi-level signal')
    plt.plot(onebit[chunk], 'C0', lw=0.5 if lowpass is not None else 2, alpha=.8, label='binary signal')
    if lowpass:
        plt.plot(sp.lfilter(*lowpass, onebit[delay:])[chunk], 'C15', lw=2, label='binary, lowpass-filtered')
    plt.legend()
    plt.tight_layout()

we can zoom in to see more clearly:

In [ ]:
show_pdm(pacman, pacman_onebit, slice(6100, 6300))

## and now the magic trick!

if we apply a simple lowpass filter to the binary multiplexed sequence we obtain a very good approximation to the multilevel additive mix!

In [ ]:
show_pdm(pacman, pacman_onebit, slice(6000, 6800), sp.butter(2, 0.15))

# towards pulse density modulation

multiplexing seems to work but of course there are some big questions:
 * how general is this method? can we use it for more than two binary waveforms?
 * shouldn't the fast oscillations produce some significant audio artifacts?
 * where is the "magic" lowpass filter? in the previous examples, we played the binary signal directly!


## averaging binary square waves

consider a discrete-time, binary square wave with a period of $P$ samples and with $D$ ones per period ($0 \le D \le P$)
 * the wave's  _duty cycle_ is $d = D/P$
 * there are $P+1$ possible values for the duty cycle: $d \in \left\{0, \frac{1}{P}, \frac{2}{P}, \ldots. \frac{P-1}{P}, 1\right\}$
 * the average of $P$ consecutive wave samples is equal to the duty cycle:
   $$
       \frac{1}{P}\sum_{k=0}^{P-1}x[n+k] = d \quad \forall n \in \mathbb{Z}
   $$

In [ ]:
params = [(4, 2, 7), (4, 3, 7), (5, 3, 6)]

def show_duty_cycles(params):
    for n, p in enumerate(params):
        plt.subplot(2,3,n+1) 
        plt.stem(square_wave(1/p[0], p[2] * p[0], 100 * p[1] / p[0], 1))
        plt.ylim(-0.2, 1.2)
        plt.title(f'duty cycle: {p[1]}/{p[0]}')

In [ ]:
show_duty_cycles(params)

## a lowpass filter is an averaging device

 * all lowpass filters work by computing a weighed _average_ of past input samples; for example:
   * the Moving Average filter literally computes the average of the past $N$ samples
     $$
         y[n] = \frac{1}{N}\sum_{k=0}^{N-1} x[n-k] = \frac{1}{N}\left(x[n] + x[n-1] + \ldots + x[n-N+1]\right)
     $$
   * the Leaky Integrator computes a recursive approximation of a moving average
     $$
         y[n] = (1-\lambda) x[n] + \lambda y[n-1], \quad |\lambda | < 1
     $$

 * let's apply a leaky integrator to the previous binary square waves:

In [ ]:
def show_averaging(params):
    N = 200
    for n, p in enumerate(params):
        x = square_wave(1 / p[0], N, 100 * p[1] / p[0], 1)
        y = sp.lfilter(*sp.butter(4, 0.1), x)
        lmb = 0.98
        y, _ = sp.lfilter([1-lmb], [1, -lmb], x, zi=np.array([0.5]))
        a = p[1] / p[0]
        plt.subplot(2,3,n+1) 
        plt.plot(x, 'green', [0, N], [a, a], 'blue', y, 'red')
        plt.ylim(-0.2, 1.2)
        plt.title(f'duty cycle: {p[1]}/{p[0]}')

In [ ]:
show_averaging(params)

## but where was the lowpass filter in the examples we played before?

<img src="img/loudspeaker_resp.png" style="float: right; width: 450px; margin: 00px 30px 20px 10px;"/>

answer: in the loudspeaker!

 * most loudspeakers exhibit a lowpass characteristic, with a steep rolloff after 20 kHz
 * the lowpass response is due to 
   * intentional design choices
   * unintentional hardware limitations (especially for cheap, small speakers)
   * the addition of safety capacitors to prevent overload (we can see the capacitor in the ZX Spectrum schematics)

<img src="img/zxspeaker.png" style="float: left; height: 200px; margin: 10px 0px 0px 100px;"/>

## audio artifacts and oversampling

 * a loudspeaker is a mediocre lowpass and it won't completely eliminate the high frequencies caused by multiplexing
 * at $F_s$ samples per second, a square wave with period $P$ has spectral peaks at multiples of $f_P = F_s/P$, independently of duty cycle
 * to avoid spurious buzzing, $f_P$ must be beyond hearing range (i.e. $f_P > 20~\text{kHz}$)
 * this requires $F_s > 20\,P~\text{kHz}$, that is, signals must be generated with an oversampling factor of at least $P$

in a ZX Spectrum, $F_s \approx 200,000$; this 10-time oversampling was fully exploited by games like Vectron

the following plot shows how in the one-bit version of the PacMan jingle, the PDM noise is mostly above 20 kHz:

In [ ]:
def compare_spectra(x_multi, x_binary, sf=SF):
    plt.subplot(211)
    for (x, p) in zip(
            (x_multi, x_binary), 
            ({'label': 'multilevel signal', 'color': 'C11', 'alpha': .8, 'lw': 4}, {'label': 'binary signal', 'color': 'C15'})):
        N = len(x) // 2
        plt.plot(np.linspace(0, sf / 2, N-1), np.abs(np.fft.fft(x))[1:N], **p)
    plt.xticks([0, sf // 4, sf // 2])
    plt.legend()

In [ ]:
compare_spectra(pacman, pacman_onebit)

# what about mixing more than two signals?

if we want to mix $M$ binary waveforms:

 * with averaging, the output will span $M-1$ intermediate levels between zero and one:
   \begin{align*}
       v_k[n] &\in \{0, 1\}, \quad k = 0, 1, \ldots, M-1 \\ \\
       \frac{1}{M}\sum_{k=0}^{M-1}v_k[n] &\in \left\{0, \frac{1}{M}, \frac{2}{M}, \ldots, \frac{M-1}{M}, 1 \right\}
   \end{align*}
 * with multiplexing:
   * the alternating portions of the output will consist of square waves with period $M$ and duty cycle between $0$ and $1$
   * to prevent buzzing artifacts, we must use an oversampling factor at least equal to $M$


<img src="img/bdbm.png" style="width: 800px; margin: 0px 0px 30px 40px;"/>

Let's try with a four-part piece:
 * with four voices the period is $P=4$
 * to push artifacts outside of hearing range we need $F_s > 80~\text{kHz}$
 * on this PC, the highest available sampling frequency is $F_s = 96~\text{kHz}$, so we're OK.

In [ ]:
bdbm = [
     (('Bb4', 4), ('Eb5', 6), ('F5', 2), ('D5', 8), (' ', 4),  
      ('Eb5', 4), ('Ab4', 4), ('Ab4', 4), ('Ab4', 8), ('G4', 4), 
      (' ', 2), ('Bb4', 2), ('D5', 2), ('Bb4', 2), ('A4', 2), ('Bb4', 2), 
      ('F4', 2), ('Bb4', 2), ('D5', 2), ('Bb4', 2), ('A4', 2), ('Bb4', 2), 
      ('Eb4', 4), ('C5', 6), ('D5', 1), ('Eb5', 1),
      ('D5', 3), ('C5', 1), ('Bb4', 3), ('C5', 1), ('F4', 3), ('A4', 1), 
      ('Bb4', 12)),

     (('G4', 4), ('G4', 4), ('A4', 4), ('Bb4', 8), (' ', 4), ('Bb4', 4), 
      ('F4', 4), ('F4', 4), ('F4', 8), (' ', 4), ('G4', 12), ('F4', 12), 
      (' ', 12), (' ', 8), ('Eb4', 4), (' ', 12)),

     ((' ', 12), ('F4', 8), (' ', 4), ('Eb4', 4), ('F4', 4), ('C3', 4), 
      ('Bb3', 4), ('D4', 4), ('Eb4', 4), (' ', 12), ('D4', 12), ('Bb3', 4), 
      ('F4', 4), ('A4', 4), ('Bb4', 4), ('G4', 4), ('Eb4', 4), ('D4', 12)),

     (('Eb3', 4), ('C3', 4), ('F3', 4), ('Bb2', 4), ('Bb3', 4), ('Ab3', 4), 
      ('G3', 4), ('F3', 4), ('Eb3', 4), ('D3', 4), ('Bb2', 4), ('Eb2', 4), 
      ('E2', 4), ('E2', 4), ('E2', 4), ('F2', 4), ('F2', 4), ('F2', 4), 
      ('G2', 4), ('A2', 4), ('F2', 4), ('Bb2', 4), ('Eb2', 4), ('F2', 4), ('Bb2', 12))
]

let's compare the sum of four independently-synthesized one-bit signals with the multiplexed one-bit synthesis

 * we can see the five levels of the additive mix and the corresponding duty cycles for each level in the multiplexed signal
 * since the first spectral line due to multiplexing is at 24 kHz, the speaker is enough as a lowpass filter 

In [ ]:
bdbm_params = {'sf': 96000, 'timing': 150}
bdbm_multi = np.sum([ BEEP(bdbm[i], **bdbm_params) for i in range(4) ], axis=0) / 4
bdbm_onebit = onebit_synth(bdbm, **bdbm_params)

In [ ]:
show_pdm(bdbm_multi, bdbm_onebit, slice(200000,201000), sp.butter(4, 0.2))

In [ ]:
multiplay([ bdbm_multi, bdbm_onebit ], ['original (5 levels)', '1-bit multiplexed', ], sf=bdbm_params['sf'])

## oversampling is essential

failing to fulfill $F_s > 20 M~\text{kHz}$ leads to horrific results as in this example where we use $F_s = 24~\text{kHz}$

 * since $P=4$, the first spectral peak due to multiplexing is at $f_P = 6~\text{kHz}$
 * we can try to remedy the situation by using an explicit lowpass filter (i.e. just the speaker by itself won't be enough)
 * of course the lowpass (whose cutoff must be below 6 kHz) will negatively affect the sound

In [ ]:
sf = 24000
bdbm_24 = onebit_synth(bdbm, timing=180, sf=sf)
multiplay([bdbm_24, sp.lfilter(*sp.ellip(6, 1, 70, 0.2), bdbm_24)], ['unfiltered multiplex', 'filtered multiplex'], 24000)

# volume control

volume envelopes can be obtained by multiplexing silent voices

In [ ]:
def fade_sqw(time, freq=1200, duty_cycle=40, sf=96000):
    LEVELS = 8
    N = int(time * sf / LEVELS)
    p = int(sf / freq + 0.5)
    duty = int(duty_cycle * p / 100)
    s = np.resize(np.r_[ np.ones(duty), np.zeros(p - duty) ], N * LEVELS)
    for n in range(1, LEVELS):
        s[n * N + n::LEVELS] = 0
    return s, sf

play(*fade_sqw(3))

# let's wrap up the manic miner story

<img src="img/manicminer.png" style="float: right; height: 150px; margin: -50px 30px 30px 30px;"/>

now that we know how to produce polyphonic binary signals, let's compare:
 * the original Manic Miner audio
 * the multiplexed mix of the two voices from the original "score":

In [ ]:
# convert original score representation (duration, pitch1, pitch2) to the format used by the 1-bit synth

mmi = [[], []]
for (d, p1, p2) in manic_miner_intro:
    if abs(p1-p2) == 1: # handle the phasing effect appropriately
        f1 = 64000 / min(p1, p2)
        f2 = 1 / (1 / f1 + 1 / 64000)
    else:
        f1, f2 = 64000 / p1, 64000 / p2
    mmi[0].append((f1, d))    
    mmi[1].append((f2, d))

In [ ]:
mm_params = {'sf': 64000, 'timing': 4}
mm_improved = onebit_synth(mmi, **mm_params)
multiplay([mm_original, mm_improved], ['original', 'multiplexed'], sf=mm_params['sf'])

by ear, the original appears to be in fact just a _clipped_ additive mix, i.e. the average of two binary voices with the intermediate values rounded to one; and, indeed:

In [ ]:
mm_multilevel = BEEP(mmi[0], **mm_params) + BEEP(mmi[1], **mm_params)
multiplay([mm_multilevel, np.round(mm_multilevel / 2), mm_original], ['sum of 2 voices (3 levels)', 'clipped sum (2 levels)', 'original'], sf=mm_params['sf'])

probably too late to claim a bug bounty though ;-)

# multiplexing standard audio signals

<img src="img/beatles.jpg" style="float: right; width: 250px; margin: -50px 30px 30px 20px;"/>

mixing by multiplexing (and lowpass filtering) works also for signals that are _not_ binary square waves

 * let's look at an example (courtesy of the Beatles multitrack bootlegs)
 * let's see why that works and under what conditions (a bit technical)

to keep things reasonable, the audio clips are bandlimited to 8 kHz and sampled at 16 kHz (i.e. oversample by a factor of two)

let's start with a simple additive mix:

In [ ]:
def read_audio_clips():
    sf, voc = wavfile.read('snd/rev_vocals.wav')
    sf, bck = wavfile.read('snd/rev_back.wav')
    voc = voc / 32768
    bck = bck / 32768
    return voc, bck, sf

In [ ]:
voc, bck, sf = read_audio_clips()
mix = (voc + bck) / 2

multiplay([voc, bck, mix], ['vocals', 'backing track', 'additive mix'], sf)

since we're operating at a low sampling frequency, the multiplexed mix is explicitly filtered using a sharp lowpass with cutoff 8 kHz

In [ ]:
mix_mpx = np.copy(bck)
mix_mpx[1::2] = voc[1::2]
mix_mpx = sp.lfilter(*sp.ellip(6, 1, 90, 0.5), mix_mpx)

multiplay([mix_mpx, mix], ['multiplexed mix', 'additive mix'], sf)

## why this works: let's do a bit of signal processing

setup: we want to mix $M$ channels $v_m[n], m=0, 1, \ldots, M-1$ <br><br>

 1. standard mix: $\displaystyle \mathbf{x} = (1/M)\sum_{m=0}^{M-1}\mathbf{v}_m$
 2. multiplexed mix: $\hat{\mathbf{x}} = \mathbf{h} \ast \mathbf{w}$, where
    * $\mathbf{w}$ is the interleaved sequence $w[n] = v_{(n \text{ mod } M)}[n]$
    * $\mathbf{h}$ is the impulse response of a lowpass filter with suitable cutoff

<br>

the question: under what conditions is $\hat{\mathbf{x}} = \mathbf{x}$ ?


## what happens in the time domain

 * we can write the multiplexed signal as the sum $\displaystyle \hat{\mathbf{x}} = \sum_{m=0}^{M-1}\mathbf{s}_m$, where $\displaystyle s_m[n] = \begin{cases} v_m[n] & \text{if } (n \text{ mod } M) = m \\ 0 & \text{otherwise} \end{cases}$
 * $\mathbf{s}_m$ is a "punctured" version of the $m$-th channel, where $M-1$ samples out of every $M$ are set to zero
 * each sequence $\mathbf{s}_m$ can be expressed equivalently as
   $$
    \mathbf{s}_m = \mathcal{S}^{m} \mathcal{U}_M \mathcal{D}_M \mathcal{S}^{-m} \mathbf{v}_m
   $$
   * $\mathcal{D}_M$ is the downsampling-by-$M$ operator (discards $M-1$ samples out of $M$)
   * $\mathcal{U}_M$ is the upsampling-by-$M$ operator (inserts $M-1$ zeros after each sample)
   * downsampling and then upsampling by $M$ in sequence has the net effect of setting to zero $M-1$ samples out of $M$
   * the delays $\mathcal{S}^{\pm m}$ are just a technicality to correctly align the sequences and can be ignored

the use of downsampling points to a necessary condtion for this scheme to work: each channel must be oversampled by at least a factor of $M$ in order to avoid aliasing!

In [ ]:
def puncturing_in_time(m, M=4, N=12):
    m = m % M
    n = np.arange(2 * N + 10)
    v = (np.sinc((n - N // 3) / N * 2) - 1) * 1.3 + 1
    mask = np.zeros_like(n)
    mask[m::M] = 1
    s = mask * v
    plt.subplot(231)
    plt.stem(v[:2*N+1], 'C0')
    plt.yticks([-1, 0, 1])
    plt.ylim([-1.2, 1.2])
    plt.title(f'$\\mathbf{{v}}_{{{m}}}$')
    plt.subplot(232)
    plt.stem(v[m:m+2*N+1], 'C7')
    plt.yticks([-1, 0, 1])
    plt.ylim([-1.2, 1.2])
    plt.title(f'$\\mathcal{{S}}^{{-{m}}}\\mathbf{{v}}_{{{m}}}$')
    plt.subplot(233)
    plt.stem(v[m:m+2*N+1:M], 'C7')
    plt.yticks([-1, 0, 1])
    plt.ylim([-1.2, 1.2])
    plt.title(f'$\\mathcal{{D}}_{{{M}}} \\mathcal{{S}}^{{-{m}}}\\mathbf{{v}}_{{{m}}}$')
    plt.subplot(234)
    plt.stem(s[m:m+2*N+1], 'C7')
    plt.yticks([-1, 0, 1])
    plt.ylim([-1.2, 1.2])
    plt.title(f'$\\mathcal{{U}}_{{{M}}} \\mathcal{{D}}_{{{M}}} \\mathcal{{S}}^{{-{m}}}\\mathbf{{v}}_{{{m}}}$')
    plt.subplot(235)
    plt.stem(s[:2*N+1], 'C1')
    plt.yticks([-1, 0, 1])
    plt.ylim([-1.2, 1.2])
    plt.title(f'$\\mathbf{{s}}_{{{m}}} = \\mathcal{{S}}^{{{m}}} \\mathcal{{U}}_{{{M}}} \\mathcal{{D}}_{{{M}}} \\mathcal{{S}}^{{-{m}}}\\mathbf{{v}}_{{{m}}}$')
    plt.tight_layout()

In [ ]:
puncturing_in_time(1, M=4)

In [ ]:
def puncture(x, M, m=0):
    m = m % M
    s = np.copy(x)
    for n in range(1, M):
        s[m+n::M] = 0
    return s

## what happens in the frequency domain

if we interpolation all signals with rate $F_s$ Hz, the corresponding spectra will be 

\begin{align*}
    \mathbf{v}_m &\rightarrow V_m(f) \\ 
    \mathbf{s}_m &\rightarrow S_m(f) = \frac{1}{M} \sum_{k=0}^{M-1} e^{j(F_s/M)mk}\, V_m\left(f - m\frac{F_s}{M}\right) \\
    \hat{\mathbf{x}} &\rightarrow \hat{X}(f) = \sum_{m=0}^{M-1} S_m(f)
\end{align*}

 * puncturing creates copies of the original spectrum at all multiples of $f_M = F_s/M$
 * if $V_m(f)$ is zero for $f > f_M$, the $M$ copies appearing in $S_m(f)$ won't overlap (no aliasing) and so, over the interval $[-f_M, f_M]$, the spectrum of the punctured signals will be simply a scaled version of the original: 
   $$
          S_m(f) = (1/M)V_m(f) \quad \text{for } f \in [-f_M, f_M]
   $$
 * if all voices have no frequency content above $f_M$, the spectrum of the multiplexed sequence will be equal to the average of the spectra of the non-punctured voices over the interval $[-f_M, f_M]$:
   $$
        \hat{X}(f) = \sum_{m=0}^{M-1} S_m(f) = (1/M)\sum_{m=0}^{M-1}V_m(f) \quad \text{for } f \in [-f_M, f_M]
   $$
 * therefore, after lowpass filtering with cutoff $f_M$, $\hat{X}(f) = X(f) \Rightarrow \hat{\mathbf{x}} = \mathbf{x}$

In [ ]:
def puncturing_in_frequency(bw, M, m=0):
    N, F = 200, 3.2
    n, f = np.arange(-N, N+1), np.arange(-int(F * N), int(F * N)+1)
    L, K = len(n), int(M * F / 2)
    v = np.sinc(n * bw / 100 / 3) ** 3
    V = np.abs(np.fft.fft(v))
    S = np.abs(np.fft.fft(puncture(v, M, m)))    
    plt.plot(f, V[f % L], 'C0', alpha=0.7, label=f'$|V_{{{m}}}(f)|$')
    if M > 1:
        plt.plot(f, S[f % L], 'C1', label=f'$|S_{{{m}}}(f)|$')
    pos, lab = np.arange(-K, K+1), []
    for d in pos:
        r = Fraction(d, M)
        if r.numerator == 0:
            lab += [r'$0$']
        else:
            a = '-' if r.numerator < 0 else ''
            a = f'{r.numerator}' if abs(r.numerator) != 1 else a
            b = '' if r.denominator == 1 else f'/{r.denominator}'
            lab += [f'${a}F_s{b}$']
    plt.xticks(L / M * pos, lab)
    plt.legend()

In [ ]:
wg.interactive(puncturing_in_frequency, bw=wg.IntSlider(min=1, max=100, value=20, step=1), M=wg.IntSlider(min=1, max=10, step=1, value=1), m=wg.fixed(0))

# why is all of this still relevant today?

an analog to digital converter performs two tasks:
   * it **samples** the analog input signal and transforms it into a discrete-time sequence
   * it **quantizes** the value of the samples over $R$ bits

the design parameters are
   * the sampling frequency $F_s$ must be at least twice the max input frequency; for audio this means $F_s > 44$ kHz
   * $R$, the number of bits per sample, determines the SNR of the digital audio, $\text{SNR} \approx 6R~\text{dB}$
   
today, the cost of A/D and D/A converters is mainly a function of $R$, and not of $F_s$
 * we would like to use as few bits per sample as possible
 * we are willing to sample much faster if that helps, since speed is cheap

# advantages of one-bit audio

 * when sampling an analog signal, the quantizer reduces to a single cheap comparator 
 * conversion back to analog can be performed by a simple lowpass filter
 * the digital data stream has no "framing" so decoding can begin anywhere in the sequence
 
the question:
 * DVD-quality audio is sampled at 48 kHz and quantized over 16 bits (SNR ~ 96 dB)
 * can we obtain the same SNR using just one bit per sample?

# can we just use PDM like before?

let's go back to mixing binary waveforms:
 * the additive mix produces signals with non-binary intermediate levels
 * the multiplexed mix yields binary fast square waves that, once averaged, reproduce the intermediate levels
 * the number of possible output levels is equal to the period $P$ of the fast square waves
 * to avoid artifacts, the required oversampling factor must be at least $P$

now let's consider DVD audio:
 * at 16 bits per sample, the signal has $2^{16} = 65536$ possible output levels
 * if we tried to encode these levels with fast square wawes, we would need $P=2^{16}$
 * the resulting one-bit audio stream would thus require a rate of $20,000 \cdot 2^{16} \approx 1.3~\text{GHz}$ !!!

this is clearly impossible

# sigma-delta

<img src="img/sigmadelta.png" style="float: right; height: 150px; margin: -10px 30px;"/>

a sigma-delta encoder is a high-speed A/D converter at one bit per sample. sigma deltas are used everywhere today -- you certainly have a couple in your cell phone!

the three components of a sigma delta encoder are

 1. the "delta" part:
    * PDM worked well when we had just a few intermediate values to encode
    * with a large oversampling factor, the difference between consecutive samples is small
    * idea: use PDM to encode **differences** (delta) instead of amplitudes
 2. the feedback part:
    * the bitstream generated by the encoder is decoded using a lowpass filter $H(z)$
    * the result is fed back in a loop to the encoder's input
    * the encoder produces each new output bit by comparing feedback with input
 4. the "sigma" part:
    * the feedback (a sum of past output bits) is compared to a **running sum** (sigma) of past input samples
    * if the difference (delta) of the sums (sigmas) is positive, the encoder output a one, otherwise a zero

## implementation

the algorithm is very simple and requires only a couple of lines of code:

In [ ]:
def sigma_delta(x, acc=0):
    ret = np.zeros(len(x))
    for n in range(0, len(x)):
        if acc >= 0:
            ret[n] = 1
            acc += x[n] - 1
        else:
            ret[n] = -1
            acc += x[n] + 1
    return ret

in this plot we can see how the encoder varies the local duty cycle so that the average of the binary waveform tracks the input

In [ ]:
N = 5000
x = 0.9 * np.cos(np.arange(N) / N * 20 * np.pi)
show_pdm(x, sigma_delta(x), slice(N-1000, N), sp.ellip(6, 0.1, 50, 0.06), delay=16)

## testing the encoder on actual audio

sigma-delta is a sophisticated form of PDM and so we need to use a large oversampling factor to make sure artifacts are inaudible

<img width="200" style="float: right; margin: -40px 10px 40px" src="img/sob.jpg">

the following excerpt from Wendy Carlos' *Switched On Brandenburgs* has been first converted to a 16-bit, 8 kHz, mono signal and then upsampled 12 times to a sampling rate of 96 kHz



In [ ]:
def interpolate(x, K):
    return K * sp.lfilter(*sp.butter(10, 1/K), np.kron(x, np.r_[1, np.zeros(K-1)]))

In [ ]:
sf, brand = wavfile.read('snd/brand1.wav')
brand = np.array(brand, dtype=float)
brand = brand - np.mean(brand)
brand = 0.9 * (brand / np.max(np.abs(brand)))

K = 12
brand = interpolate(brand, K)

In [ ]:
play(brand, 96000)

as a comparison point, this is what happens if we simply scale down the samples to a resolution of one bit:

In [ ]:
play(np.ceil(brand), 96000)

we can now apply sigma-delta encoding and listen to the result; since the PDM "period" is time-varying, we need to lowpass filter the bitstream explicitly to completely eliminate the audio artifacts

in terms of SNR, the result is approximately equivalent to a PCM signal with 12 bits per sample

In [ ]:
brand_sd = sigma_delta(brand)
play(sp.lfilter(*sp.ellip(6, 1, 50, 0.08), brand_sd), 96000)

finally, here we can see how the lowpass transforms the time-varying pulse density into a multilevel signal

In [ ]:
show_pdm(brand, brand_sd, slice(200000,201000), sp.ellip(6, 0.5, 50, 0.08), delay=16)

# DSD

<img src="img/dsd.png" style="float: right; width: 200px; margin: 00px 30px;"/>

Direct Stream Digital (DSD) is the digital format used in Super Audio CD (SACD):

 * 64-time oversampling wrt CD audio ($F_s = 2.8224~\text{MHz}$)
 * 1 bit/sample, data rate: 5.6 Mbit/s
 * SNR: 120 dB